# Why were 14 Q1 rows set aside, and how does the auditor know nothing else went?

Week 1, Wednesday afternoon, the second case, in pairs. Anand's auditor writes: "Your log says 14 Q1 rows were dropped. Why those 14, and how do I know nothing
else went with them?" The log is the set-aside log the day's pass wrote. The pass is the chain of steps
that cleans the raw export from the ERP, the enterprise resource planning system Finance books orders
in, and its set-aside log gives every row it removed a reason and the line of its twin, the kept row it
copies.

**Who needs the answer.** The auditor signs off Kalpa's Q1 figures and may ask next quarter why any row went, and Anand Iyer, the
finance controller, needs the answer to hold. An answer the auditor cannot follow costs the team a week
of questions, and one that later proves wrong costs the team the auditor's trust in everything else it
sends.

**The questions on the way.**

1. Which count gives the auditor's 14?
2. Is every one of the 14 a copy?
3. Which of the 14 rows carry the rupees?
4. Why was one copy chosen over its twin?
5. What evidence can the auditor sign to show nothing else went?

Answer from the log and the file, one question at a time. One of the pair drives; the other plays
the auditor and asks the next question only when the check passes.

Each question has lettered choices above a placeholder such as `__TODO1__`. Replace it with the
code of the option you choose, run it, then run the check cell. Run as shipped, the notebook stops
at the first placeholder with a `NameError`, which is expected. Post your five letters when every
check passes.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

import csv, json, math
from collections import Counter
from datetime import date

DATA = kit.data_dir()
ORDERS_CSV = DATA / "C2_W01_D03_orders_STUDENT.csv"
BOOKS_Q1 = 19000000        # Anand's Q1 to the rupee, which his analyst sends on request

def read_orders(path=ORDERS_CSV):
    """Every row of an export as a dictionary of text, with the file line it came from."""
    with open(path, newline="", encoding="utf-8") as f:
        return [dict(row, line=n) for n, row in enumerate(csv.DictReader(f), start=2)]

def convert(value):
    """A whole number of rupees, or None with the reason, so a failure is counted, never hidden."""
    try:
        return int(value), ""
    except (TypeError, ValueError):
        return None, "amount does not convert to a whole number"

def Q(rows, quarter):
    """Rupees in a quarter over the amounts that convert."""
    return sum(convert(r["amount"])[0] or 0 for r in rows if r["quarter"] == quarter)

def identity_rule(rows, key="order_id"):
    """One row per order: the first copy whose amount converts stays, and every other row is logged."""
    groups = {}
    for r in rows:
        groups.setdefault(r[key], []).append(r)
    kept, log = [], []
    for k, rs in groups.items():
        valid = [r for r in rs if convert(r["amount"])[0] is not None]
        keep = (valid or rs)[0]
        kept.append(keep)
        for r in rs:
            if r is keep:
                continue
            differs = [f for f in r if f != "line" and r[f] != keep[f]]
            reason = ("copy whose amount does not convert; its twin carries the value"
                      if convert(r["amount"])[0] is None else "second copy of the order")
            if differs and "amount" not in differs:
                reason += "; differs on " + ", ".join(differs)
            log.append({"line": r["line"], "order_id": k, "quarter": r["quarter"], "segment": r["segment"],
                        "amount": r["amount"], "rule": key, "reason": reason, "kept_line": keep["line"]})
    return kept, log

def clean_pass(raw):
    """The day's pass: the identity rule, then conversion with a rejects log, then flags."""
    kept, set_aside = identity_rule(raw)
    clean, rejects, flags = [], [], []
    for r in kept:
        value, reason = convert(r["amount"])
        if value is None:
            rejects.append({"line": r["line"], "order_id": r["order_id"], "field": "amount",
                            "value": r["amount"], "reason": reason})
            continue
        if not r["status"]:
            flags.append({"line": r["line"], "order_id": r["order_id"], "field": "status",
                          "decision": "keep and flag", "why": "booked revenue; its fate is unknown"})
        clean.append(dict(r, amount=value))
    q2 = [r for r in clean if r["quarter"] == "Q2"]
    top = max(q2, key=lambda r: r["amount"])
    flags.append({"line": top["line"], "order_id": top["order_id"], "field": "amount",
                  "decision": "keep and flag", "why": "the largest Q2 order, a real Business account; shown both ways"})
    return clean, set_aside, rejects, flags

def per_customer(rows, quarter, segment):
    rs = [r for r in rows if r["quarter"] == quarter and r["segment"] == segment]
    return len(rs) / len({r["customer_id"] for r in rs})

raw = read_orders()
clean, set_aside, rejects, flags = clean_pass(raw)
raw_by_line = {r["line"]: r for r in raw}
q1_aside = [e for e in set_aside if e["quarter"] == "Q1"]
print(len(clean), "orders kept; the set-aside log, the rejects log and the flags log are loaded")

In [ ]:
kit.side_by_side(
    kit.ladder(["Chapters 1 to 6: how is the pass built?", "The escalated case: can you run it alone?",
            "The second case: why were 14 rows set aside?"], lit=2, show=False),
    kit.vflow(["1. Which count gives the 14?\nthe auditor's first question",
               "2. Is every one of the 14 a copy?\nthe evidence of a copy",
               "3. Which rows carry the rupees?\nlaid out for the auditor",
               "4. Why did one copy stay?\nthe log's reasons",
               "5. What can the auditor sign?\nthe evidence that nothing else went"], lit=0, show=False),
)

## Question 1. Which count gives the auditor's 14?

At work, the first reply to an auditor reproduces the number she quotes, before any explanation.

In [ ]:
q1_in = sum(1 for r in raw if r["quarter"] == "Q1")
q1_kept = sum(1 for r in clean if r["quarter"] == "Q1")
# TODO 1. Which count answers the auditor's first question, which 14?
#   a) len(raw) - len(clean)
#   b) len({r["order_id"] for r in raw if r["quarter"] == "Q1"})
#   c) q1_in - q1_kept
#   d) len(rejects)
answer_1 = __TODO1__
print("Question 1: done")

In [ ]:
kit.check("your count is the auditor's 14, and the log lists that many Q1 rows", answer_1 == 14 == len(q1_aside), f"{answer_1}")
kit.columns(["Q1"], [("rows in", [q1_in]), ("orders kept", [q1_kept]), ("set aside", [len(q1_aside)])],
            title="Q1 rows in, kept and set aside")

## Question 2. Is every one of the 14 a copy?

At work, every row that leaves a file needs its own evidence of why it left, before anyone signs.

In [ ]:
kept_raw = [raw_by_line[r["line"]] for r in clean]
aside_raw = [raw_by_line[e["line"]] for e in q1_aside]

def has_twin(row, others):
    # TODO 2. Which test says a row has a twin among the others, the evidence that it was a copy?
    #   a) any(o["amount"] == row["amount"] for o in others)
    #   b) any(o["customer_id"] == row["customer_id"] for o in others)
    #   c) row["line"] > len(others)
    #   d) any(o["order_id"] == row["order_id"] for o in others)
    return __TODO2__
print("Question 2: done")

In [ ]:
kit.check("the test finds a twin among the kept orders for every one of the 14", all(has_twin(r, kept_raw) for r in aside_raw))
kit.check("and finds none for a kept order among the other kept orders",
          not any(has_twin(r, [o for o in kept_raw if o is not r]) for r in kept_raw))

## Question 3. Which of the 14 rows carry the rupees?

At work, an auditor with an hour to spend looks first where the money sits.

In [ ]:
segs = ["Business", "Retail-Plus", "Retail-Core"]
rows_by = [sum(1 for e in q1_aside if e["segment"] == s) for s in segs]
rupees_by = [sum(convert(e["amount"])[0] or 0 for e in q1_aside if e["segment"] == s) for s in segs]
# TODO 3. How should the rupees be laid out for the auditor?
#   a) [sum(rupees_by)]
#   b) list(zip(segs, rows_by, rupees_by))
#   c) [(s, v) for s, v in zip(segs, rupees_by)]
#   d) [(s, n, v / Q(clean, "Q2")) for s, n, v in zip(segs, rows_by, rupees_by)]
shown = __TODO3__
print("Question 3: done")

In [ ]:
def lays_out(table):
    try:
        return (sorted(x[0] for x in table) == sorted(segs) and sum(x[1] for x in table) == len(q1_aside)
                and sum(x[2] for x in table) == sum(convert(e["amount"])[0] or 0 for e in q1_aside))
    except (TypeError, IndexError):
        return False

kit.check("one line per segment, the rows adding to the 14 and the rupees to all set aside in Q1", lays_out(shown))
kit.bars(list(zip(segs, rupees_by)), fmt=kit.rupees, title="Rupees in the 14 Q1 rows, by segment")
kit.bars(list(zip(segs, rows_by)), title="Rows among the 14, by segment")

## Question 4. Why was one copy chosen over its twin?

At work, whoever reruns a pass months later has only the log to say what the rule decided.

In [ ]:
by_id = {}
for r in raw:
    by_id.setdefault(r["order_id"], []).append(r)
kept_by_id = {r["order_id"]: r for r in clean}
differing = {k: rs for k, rs in by_id.items() if len(rs) > 1
             and len({tuple((f, v) for f, v in r.items() if f != "line") for r in rs}) > 1}

def fields_differ(rs):
    return [f for f in rs[0] if f != "line" and rs[0][f] != rs[1][f]]

# TODO 4. For each pair whose copies differ, which log line lets the auditor follow the choice?
#   a) {"order_id": k, "kept": kept_by_id[k]["line"], "differs": fields_differ(rs)}
#   b) {"order_id": k, "note": "the pair shares an id"}
#   c) {"order_id": k, "amount": sum(convert(r["amount"])[0] or 0 for r in rs) / len(rs)}
#   d) {"order_id": k, "deleted": [r["line"] for r in rs]}
log_lines = {k: __TODO4__ for k, rs in differing.items()}
print("Question 4: done")

In [ ]:
kit.check("two pairs differ between their copies", len(differing) == 2)
kit.check("each log line names the row that stayed", all(e.get("kept") in {r["line"] for r in clean} for e in log_lines.values()))
kit.check("and the fields its copies disagree on", all(e.get("differs") == fields_differ(differing[k]) for k, e in log_lines.items()))

## Question 5. What evidence can the auditor sign to show nothing else went?

At work, a sign-off is the point after which nobody reopens the numbers, so it rests on evidence.

In [ ]:
def q1_rows(rows):
    return [r for r in rows if r["quarter"] == "Q1"]

def rows_tie(clean_rows, aside_rows):
    return len(q1_rows(raw)) == len(q1_rows(clean_rows)) + len(q1_rows(aside_rows))

def rupees_tie(aside_rows):
    return Q(raw, "Q1") - Q(aside_rows, "Q1") == BOOKS_Q1

def colleague_pass(rows):
    """Chapter 6's colleague: repeated ids out first, keeping the first copy, then conversion."""
    seen, first, aside = set(), [], []
    for r in rows:
        (aside if r["order_id"] in seen else first).append(r)
        seen.add(r["order_id"])
    good = [dict(r, amount=convert(r["amount"])[0]) for r in first if convert(r["amount"])[0] is not None]
    return good, aside + [r for r in first if convert(r["amount"])[0] is None]

def evidence(c, a):
    # TODO 5. Which evidence shows nothing else went, and would have caught a pass that kept the wrong copy?
    #   a) rows_tie(c, a)
    #   b) round(Q(c, "Q1") / 1e7, 1) == round(BOOKS_Q1 / 1e7, 1)
    #   c) rows_tie(c, a) and rupees_tie(a)
    #   d) len(a) == len(raw) - len(c)
    return __TODO5__
print("Question 5: done")

In [ ]:
col_clean, col_aside = colleague_pass(raw)
kit.check("the evidence holds for this pass", evidence(clean, set_aside))
kit.check("and would have caught the colleague's pass, which kept the unreadable copy", not evidence(col_clean, col_aside))
kit.flow(["the question\nwhy 14 rows", "the count\nQ1 in less kept", "the twins\none for each",
          "the rupees\nby segment", "the signature\nrows and rupees"], lit=4, title="The auditor's walk through the log")
kit.check_summary()